# 🚀 Huấn luyện mô hình MaxMViT-MLP trên Google Colab

Notebook này được thiết kế để huấn luyện mô hình **MaxMViT-MLP (GMU Fusion)** trên tập dữ liệu tiếng Việt **ViSEC** bằng Google Colab (GPU T4/V100/A100).

> **Lưu ý quan trọng trước khi bắt đầu:**
> 1. Vào menu **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)**.
> 2. Tại mục **Hardware accelerator (Bộ tăng tốc phần cứng)**, chọn **T4 GPU** (hoặc GPU khác) rồi nhấn **Save**.

### Bước 1: Kiểm tra cấu hình GPU

In [ ]:
!nvidia-smi

### Bước 2: (Tùy chọn) Kết nối Google Drive để lưu checkpoint vĩnh viễn
Khi Colab hết phiên làm việc, dữ liệu trên máy ảo sẽ bị xóa. Mount Google Drive giúp tự động sao lưu checkpoint an toàn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục sao lưu trên Drive của bạn
!mkdir -p /content/drive/MyDrive/MaxMViT_Checkpoints

### Bước 3: Clone mã nguồn mới nhất từ GitHub

In [ ]:
# Xóa thư mục cũ nếu có để đảm bảo lấy code mới nhất
!rm -rf MaxMViT-MLP-SER
!git clone https://github.com/Huu2412/MaxMViT-MLP-SER.git
%cd MaxMViT-MLP-SER

### Bước 4: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q timm librosa soundfile datasets pyyaml scikit-learn seaborn matplotlib thop

### Bước 5: Bắt đầu huấn luyện mô hình

Lệnh dưới đây sẽ chạy huấn luyện với cấu hình tối ưu `configs/visec_optimized.yaml`:
- **Phân chia dữ liệu:** 80% Train (4,224 mẫu), 10% Val (528 mẫu), 10% Test (528 mẫu).
- **Mô hình:** MaxViT + MViTv2 + GMU Fusion + Multi-task Region Recognition.
- **Tự động đo lường:** Sau khi train xong, mô hình tốt nhất sẽ tự động được đánh giá trên tập **Test độc lập** và xuất file `_report.txt` cùng `_predictions.json`.

In [ ]:
# Chạy huấn luyện trên GPU Colab
!python train.py --config configs/visec_optimized.yaml

### Bước 6: Sao lưu Checkpoints & Báo cáo vào Google Drive
Sau khi train xong, copy toàn bộ checkpoints và kết quả sang Google Drive để lưu trữ lâu dài.

In [ ]:
!cp -r checkpoints/* /content/drive/MyDrive/MaxMViT_Checkpoints/
print("✅ Đã sao lưu toàn bộ checkpoints sang Google Drive:")
!ls -la /content/drive/MyDrive/MaxMViT_Checkpoints/

### Bước 7: Xem nhanh kết quả đánh giá trên tập Test

In [ ]:
import glob

reports = glob.glob('checkpoints/*_report.txt')
if reports:
    print(f"=== KẾT QUẢ ĐÁNH GIÁ ({reports[0]}): ===\n")
    with open(reports[0], 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Chưa tìm thấy file report. Hãy chạy huấn luyện trước.")